### install et chemins

In [1]:
!nvidia-smi
!pip install -q faster-whisper

Sun Nov 23 16:53:23 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

AUDIO_DIR = "/content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_audios"       # là où tu as mis les mp3
TRANSCRIPTS_DIR = "/content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_transcripts"  # un json par audio

os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)

# Whisper !

In [4]:
from faster_whisper import WhisperModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

# Modèle : large-v3 si possible, sinon medium pour être plus léger
MODEL_NAME = "large-v3"

model = WhisperModel(
    MODEL_NAME,
    device=device,        # "cuda" en pratique
    compute_type="float16"
)

Device utilisé : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

## import MP3s

In [5]:
import glob
import json

audio_files = sorted(glob.glob(os.path.join(AUDIO_DIR, "*.mp3")))
print("Nombre de fichiers audio trouvés :", len(audio_files))

Nombre de fichiers audio trouvés : 97


## exécution ?

In [6]:
for i, audio_path in enumerate(audio_files, start=1):
    base = os.path.basename(audio_path)           # ex: "vXw8v0JcPng.mp3"
    stem = os.path.splitext(base)[0]             # ex: "vXw8v0JcPng"
    json_path = os.path.join(TRANSCRIPTS_DIR, stem + ".json")

    # Si tu veux pouvoir relancer sans tout recommencer :
    if os.path.exists(json_path):
        print(f"[{i}/{len(audio_files)}] déjà transcrit, on saute : {base}")
        continue

    print(f"[{i}/{len(audio_files)}] Transcription de : {base}")

    # Transcription
    segments, info = model.transcribe(
        audio_path,
        language="fr",      # à adapter si besoin
        beam_size=5,
        vad_filter=True
    )

    # Recomposition en une seule chaîne de caractères
    full_text = "".join(segment.text for segment in segments)

    # Construction de l’objet à sauver
    data = {
        "audio_file": audio_path,
        "audio_id": stem,   # souvent = id YouTube
        "text": full_text,
    }

    # Sauvegarde JSON dans Drive
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"  → JSON écrit : {json_path}")

print("Transcription terminée.")

[1/97] Transcription de : -0p6pVBtw2M.mp3
  → JSON écrit : /content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_transcripts/-0p6pVBtw2M.json
[2/97] Transcription de : -VLk0KbMvQ4.mp3
  → JSON écrit : /content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_transcripts/-VLk0KbMvQ4.json
[3/97] Transcription de : -v2Bl2npJ9E.mp3
  → JSON écrit : /content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_transcripts/-v2Bl2npJ9E.json
[4/97] Transcription de : 0QVRaQusEto.mp3
  → JSON écrit : /content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_transcripts/0QVRaQusEto.json
[5/97] Transcription de : 1h1qXJQt414.mp3
  → JSON écrit : /content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_transcripts/1h1qXJQt414.json
[6/97] Transcription de : 2D847F69xZA.mp3
  → JSON écrit : /content/drive/MyDrive/fichiers_colab/histosef/bardella/whisper_transcripts/2D847F69xZA.json
[7/97] Transcription de : 2S1Jo7D_6JA.mp3
  → JSON écrit : /content/drive/MyDrive/fichie